# Modeling T1-T3


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skrub import TableReport
from catboost import CatBoostRegressor
import joblib, os
import preprocess
import explore
import model
from sklearn.preprocessing import PowerTransformer

os.environ['PYTHONWARNINGS'] = 'ignore'
MODEL_DIR = os.path.abspath('../models')

In [2]:
# Running through preprocessing steps
# Load raw data
df_im, df_cl = explore.load_data()

# Clean datasets
_im_id_cols = ['Patient', 'Timepoint', 'Date']
df_im_vis   = preprocess.clean_im(df_im)
df_cl_vis   = preprocess.clean_cl(df_cl)

# Immunological: drop >25% NaN columns, remove confirmed outliers
_im_nan_frac = df_im_vis.drop(columns=_im_id_cols).isna().mean()
_im_high_nan = _im_nan_frac[_im_nan_frac > 0.25].index.tolist()
df_im_mod    = df_im_vis.drop(columns=_im_high_nan)
df_im_mod    = preprocess.remove_outlier_observations(df_im_mod)

# Clinical: drop >25% NaN columns
_cl_id_cols  = ['Patient', 'Timepoint', 'date', 'measurement_timepoint']
_cl_nan_frac = df_cl_vis.drop(columns=_cl_id_cols).isna().mean()
_cl_high_nan = _cl_nan_frac[_cl_nan_frac > 0.25].index.tolist()
df_cl_mod    = df_cl_vis.drop(columns=_cl_high_nan)

print(f"df_im_mod : {df_im_mod.shape}")
print(f"df_cl_mod : {df_cl_mod.shape}")


  [1] Dropping excluded columns and renaming Messdatum to Date
  Dropped 44 columns
  [2] Dropping known empty rows
  Dropping 7 rows at index: [823, 824, 825, 826, 827, 828, 78]
  Patient IDs in dropped rows: [30.0]
  Unique patients remaining: 264
  [3] Replacing German NaN markers
  Neu_CD25+: replaced 161 null markers
  BAS_CD25+: replaced 18 null markers
  Mo_CD25+: replaced 109 null markers
  T_CD25hi: replaced 5 null markers
  TC_CD25hi: replaced 790 null markers
  TH_CD25hi: replaced 5 null markers
  B_CD25+: replaced 7 null markers
  B_CD25hi: replaced 810 null markers
  NK_CD25+: replaced 150 null markers
  Neu_HLADR+: replaced 27 null markers
  Eos_HLADR+: replaced 567 null markers
  Mo_HLADRhi: replaced 4 null markers
  Mo2_HLADR+: replaced 192 null markers
  Mo3_HLADR+: replaced 73 null markers
  Mo1_HLADRhi: replaced 11 null markers
  Mo2_HLADRhi: replaced 230 null markers
  Mo3_HLADRhi: replaced 90 null markers
  TC_HLADR+: replaced 7 null markers
  T_HLADRhi: replaced 1

In [ ]:
# Constructing targets:
print('\nConstructing regression targets from clinical data')
pain_targets       = model.construct_datasets_targets(df_cl_mod, 'pain_scale',      [1, 3])
targets_under_load = model.construct_datasets_targets(df_cl_mod, 'pain_under_load', [1, 3])

In [ ]:
# Plotting target distributions:
target_frames = {
    'pain_scale':      pain_targets,
    'pain_under_load': targets_under_load,
}

fig, axes = plt.subplots(2, len(target_frames), figsize=(5 * len(target_frames), 8))
colors = sns.color_palette('mako', len(target_frames))

for col_idx, (name, tdf) in enumerate(target_frames.items()):
    prefix  = name.replace('_scale', '')   # matches construct_datasets_targets naming
    red_col = f'{prefix}_reduction'
    pct_col = f'{prefix}_reduction_pct'

    # Absolute reduction
    ax0 = axes[0, col_idx]
    sns.histplot(tdf[red_col].dropna(), kde=True, ax=ax0,
                 color=colors[col_idx], bins=20)
    ax0.set_title(f'{name}\nAbsolute reduction (T1−T3)')
    ax0.set_xlabel('Reduction')

    # Percent reduction
    ax1 = axes[1, col_idx]
    sns.histplot(tdf[pct_col].dropna(), kde=True, ax=ax1,
                 color=colors[col_idx], bins=20)
    ax1.set_title(f'{name}\nPercent reduction (%)')
    ax1.set_xlabel('Reduction (%)')

plt.suptitle('Target Distributions (T1 - T3)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()



Constructing regression targets from clinical data

  Targets: 'pain_scale'  (T1 → T2)
  Patients with T1 values    : 189
  Patients with T2 values    : 176
  Eligible (non-NaN both, n)    : 142
  Eligible patient IDs          : [1, 5, 6, 10, 12, 15, 16, 17, 18, 19, 20, 21, 22, 23, 29, 34, 58, 72, 76, 91, 92, 93, 94, 101, 104, 105, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 131, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 153, 154, 155, 157, 158, 159, 161, 163, 164, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 214, 215, 218, 220, 221, 222, 223, 224, 225, 226, 229, 230, 232, 234, 237, 239, 240, 245, 246, 248, 250, 253, 256, 258, 259, 269, 270, 271, 272, 273]

  Target distributions:
    pain_reduction                              mean=1.349  

: 

In [ ]:
# Visualizing power-transformed target distributions (on copies)

fig, axes = plt.subplots(2, len(target_frames), figsize=(5 * len(target_frames), 8))
colors = sns.color_palette('mako', len(target_frames))

for col_idx, (name, tdf) in enumerate(target_frames.items()):
    prefix  = name.replace('_scale', '')
    red_col = f'{prefix}_reduction'
    pct_col = f'{prefix}_reduction_pct'

    tdf_viz = tdf[[red_col, pct_col]].copy()
    pt_viz  = PowerTransformer(method='yeo-johnson', standardize=True)
    tdf_viz[[red_col, pct_col]] = pt_viz.fit_transform(tdf_viz)

    ax0 = axes[0, col_idx]
    sns.histplot(tdf_viz[red_col].dropna(), kde=True, ax=ax0,
                 color=colors[col_idx], bins=20)
    ax0.set_title(f'{name}\nAbsolute reduction (T1−T3) — transformed')
    ax0.set_xlabel('Reduction (transformed)')

    ax1 = axes[1, col_idx]
    sns.histplot(tdf_viz[pct_col].dropna(), kde=True, ax=ax1,
                 color=colors[col_idx], bins=20)
    ax1.set_title(f'{name}\nPercent reduction (%) — transformed')
    ax1.set_xlabel('Reduction (%) (transformed)')

plt.suptitle('Target Distributions after Power Transform (T1 - T3)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## Dataset Overview

In [ ]:
# Construct model datasets for each target type:
_unique_targets = {
    'pain_reduction':            pain_targets,
    'pain_under_load_reduction': targets_under_load,
}

model_datasets = {}
for tgt, tdf in _unique_targets.items():
    model_datasets[tgt] = model.create_model_datasets(
        df_cl_mod, df_im_mod, tdf, timepoints=[1, 2]
    )
model_datasets['pain_reduction_pct'] = model_datasets['pain_reduction']


Constructing datasets for modeling:

Constructing regression targets from clinical data

  Targets: 'pain_scale'  (T1 → T2)
  Patients with T1 values    : 189
  Patients with T2 values    : 176
  Eligible (non-NaN both, n)    : 142
  Eligible patient IDs          : [1, 5, 6, 10, 12, 15, 16, 17, 18, 19, 20, 21, 22, 23, 29, 34, 58, 72, 76, 91, 92, 93, 94, 101, 104, 105, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 131, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 153, 154, 155, 157, 158, 159, 161, 163, 164, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 214, 215, 218, 220, 221, 222, 223, 224, 225, 226, 229, 230, 232, 234, 237, 239, 240, 245, 246, 248, 250, 253, 256, 258, 259, 269, 270, 271, 272, 273]

  Target distributions:
    pain_reduction     

In [ ]:
print('TableReport of dataset for modeling, including targets:')
TableReport(model_datasets['pain_reduction_pct'], max_plot_columns=180)
TableReport(model_datasets['pain_under_load_reduction'], max_plot_columns=180)

In [ ]:
# Correlation between targets and features:
print(f"\n{'─'*55}")
print("  Feature–target Pearson correlations (top 10, combined dataset):")
for tgt, df_comb in model_datasets.items():
    id_like = ['Patient', 'Timepoint']
    num_cols = [c for c in df_comb.select_dtypes(include='float64').columns
                if c not in id_like]
    corrs = df_comb[num_cols].corrwith(df_comb[tgt]).drop(index=tgt, errors='ignore')
    corrs = corrs.dropna().abs().sort_values(ascending=False).head(20)
    print(f"\n  {tgt}:")
    print(corrs.to_string())

## Baseline CatBoost

In [4]:

print('\n Running baseline CatBoost — pain_reduction, pain_reduction_pct, pain_under_load_reduction')

_pt = PowerTransformer(method='yeo-johnson', standardize=True)

baseline_results = {}
for tgt, df_comb in model_datasets.items():
    res_comb, mdl_comb, X_comb, ypred_comb = model.run_catboost_regressor(
        df_comb, tgt, 'Combined T1−T2 diff', target_transformer=_pt)
    baseline_results[tgt] = (res_comb, mdl_comb, X_comb, ypred_comb)

for tgt, (res, *_) in baseline_results.items():
    model.print_regression_summary({'Combined': res}, tgt)




 Running baseline CatBoost — pain_reduction, pain_reduction_pct, pain_under_load_reduction

  CatBoost Regressor Baseline — Combined T1−T2 diff
  Target : pain_reduction
  Samples: 112,  Features: 92
  CV     : 5-fold × 5 repeats = 25 fits
  Fold  1: MAE=2.441  MSE=9.328  RMSE=3.054  R²=-0.202
  Fold  2: MAE=1.684  MSE=4.017  RMSE=2.004  R²=-0.345
  Fold  3: MAE=1.365  MSE=2.989  RMSE=1.729  R²=-0.303
  Fold  4: MAE=2.357  MSE=8.046  RMSE=2.837  R²=-0.691
  Fold  5: MAE=1.294  MSE=2.266  RMSE=1.505  R²=-0.346
  Fold  6: MAE=2.116  MSE=7.205  RMSE=2.684  R²=-0.114
  Fold  7: MAE=1.767  MSE=4.582  RMSE=2.140  R²=-0.137
  Fold  8: MAE=1.802  MSE=4.421  RMSE=2.103  R²=-0.233
  Fold  9: MAE=1.403  MSE=3.492  RMSE=1.869  R²=-0.172
  Fold 10: MAE=1.509  MSE=3.923  RMSE=1.981  R²=-0.343
  Fold 11: MAE=1.632  MSE=4.761  RMSE=2.182  R²=-0.230
  Fold 12: MAE=1.723  MSE=3.822  RMSE=1.955  R²=-0.376
  Fold 13: MAE=1.904  MSE=6.591  RMSE=2.567  R²=-0.174
  Fold 14: MAE=1.506  MSE=3.381  RMSE=1.839 

## CatBoost Model - RENT, Nested CV + Optuna Hyperparameter Tuning

### Target: pain_reduction 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print('\n1.1: CatBoost (Nested CV + RENT + Optuna) — pain_reduction (T1-T2)')
_pt = PowerTransformer(method='yeo-johnson', standardize=True)

cb_red_results, cb_red_model, cb_red_X, cb_red_ypred, cb_red_model_params, cb_red_freq = \
    model.run_advanced_catboost_rent(
        model_datasets['pain_reduction'],
        target_col='pain_reduction',
        target_transformer=_pt,
    )

# Save model and feature matrix to run SHAP
cb_red_model.save_model(os.path.join(MODEL_DIR, 'cb_red_model.cbm'))
joblib.dump(cb_red_X, os.path.join(MODEL_DIR, 'cb_red_X.pkl'))
print(' Saved Catboost model (pain_reduction target) cb_red_model.cbm and cb_red_X.pkl to', os.path.abspath(MODEL_DIR))



1.1: CatBoost (Nested CV + RENT + Optuna) — pain_reduction (T1-T2)

  CatBoost + Optuna + RENT — pain_reduction
  n=112, p=92, τ₃=0.975
  Outer 4×5=20 | Inner 4×5=20 | RENT & model trials=20 | K=100

─────────────────────────────────────────────────────────────────
  Outer fold 1/20
─────────────────────────────────────────────────────────────────
  [Step 1] RENT HP tuning (20 trials, 75-25 split)
  Best RENT trial: RMSE=1.2408  {'C': 9.674929260877795, 'l1_ratio': 0.10703740332416371, 'tau_1': 0.7345459578579765, 'tau_2': 0.745242233928874}
  [Step 2] RENT on full X_train with best HPs
  RENT: 23/92 features — ['Basophils_t2_minus_t1', 'Monocytes_t2_minus_t1', 'T4:T8 ratio_t2_minus_t1', 'TREGs_t2_minus_t1', 'mDC-1_t2_minus_t1', 'Mo1_t2_minus_t1', 'Neu_CD25+_t2_minus_t1', 'TC_CD25+_t2_minus_t1']...
  [Step 3] CatBoost HP tuning (20 trials, 20 inner folds each)
    Trial   1/20: RMSE=0.9981  {'depth': 4, 'learning_rate': 0.001984704491702059, 'l2_leaf_reg': 5.15769677759593, 'bagging_t

[W 2026-03-10 18:39:03,819] Trial 10 failed with parameters: {'C': 9.79646900766545, 'l1_ratio': 0.9796214208607704, 'tau_1': 0.6004087589439248, 'tau_2': 0.6057522179965803} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "p:\UK_Erlangen\Student_folders\Muna Ahmed Farah - IMMO-LDRT01\master-thesis\notebooks\../src\model.py", line 573, in rent_objective
    probe.fit(X_tr[sel_cols], y_tr)
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\catboost\core.py", line 5873, in fit
    return self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline,
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\catboost\core.py", line 2410, in _fit
    self._train(
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\catboost\core.p

KeyboardInterrupt: 

In [ ]:
# Shap on saved model 
cb_red_model = CatBoostRegressor()
cb_red_model.load_model(os.path.join(MODEL_DIR, 'cb_red_model.cbm'))
cb_red_X = joblib.load(os.path.join(MODEL_DIR, 'cb_red_X.pkl'))

print('\n SHAP — CatBoost (pain_reduction, T1→T2)')
cb_red_shap = model.plot_shap_regressor(
    cb_red_model, cb_red_X, 'CatBoost — pain_reduction (T1→T2)')


### Target: pain_reduction_pct

In [ ]:
print('\n1.2: CatBoost (Nested CV + RENT + Optuna) — pain_reduction_pct (T1-T2)')
_pt = PowerTransformer(method='yeo-johnson', standardize=True)

cb_pct_results, cb_pct_model, cb_pct_X, cb_pct_ypred, \
cb_pct_model_params, cb_pct_freq = \
    model.run_advanced_catboost_rent(
        model_datasets['pain_reduction_pct'],
        target_col='pain_reduction_pct',
        target_transformer=_pt,
    )

# — Save model and feature matrix so SHAP can be run later without retraining —
cb_pct_model.save_model(os.path.join(MODEL_DIR, 'cb_pct_model.cbm'))
joblib.dump(cb_pct_X, os.path.join(MODEL_DIR, 'cb_pct_X.pkl'))
print(' Saved cb_pct_model.cbm and cb_pct_X.pkl to', os.path.abspath(MODEL_DIR))



1.2: CatBoost (Nested CV + RENT + Optuna) — pain_reduction_pct (T1-T2)

  CatBoost + Optuna + RENT — pain_reduction_pct
  n=112, p=92, τ₃=0.95
  Outer 4×5=20 | Inner 4×5=20 | RENT & model trials=20 | K=100

─────────────────────────────────────────────────────────────────
  Outer fold 1/20
─────────────────────────────────────────────────────────────────
  [Step 1] RENT HP tuning (20 trials, 75-25 split)
  Best RENT RMSE=1.0984  {'C': 9.405482330548738, 'l1_ratio': 0.4737271110631235, 'tau_1': 0.6279131642554363, 'tau_2': 0.6002810628455587}
  [Step 2] RENT on full X_train with best HPs
  RENT Selected : 16/92 features — ['Monocytes_t2_minus_t1', 'T4:T8 ratio_t2_minus_t1', 'mDC-1_t2_minus_t1', 'Neu_CD25+_t2_minus_t1', 'TC_CD25+_t2_minus_t1', 'B_CD25+_t2_minus_t1', 'NK_CD25+_t2_minus_t1', 'Mo_HLADR+_t2_minus_t1']...
  [Step 3] CatBoost HP tuning with 20 trials, on 20 inner folds each)
    Trial   1/20: RMSE=1.0187  {'depth': 5, 'learning_rate': 0.02984056739294969, 'l2_leaf_reg': 7.376

ValueError: too many values to unpack (expected 6)

In [ ]:
# SHAP plot on saved model
cb_pct_model = CatBoostRegressor()
cb_pct_model.load_model(os.path.join(MODEL_DIR, 'cb_pct_model.cbm'))
cb_pct_X = joblib.load(os.path.join(MODEL_DIR, 'cb_pct_X.pkl'))

print('\nSHAP — CatBoost (pain_reduction_pct, T1-T2)')
cb_pct_shap = model.plot_shap_regressor(
    cb_pct_model, cb_pct_X, 'CatBoost — pain_reduction_pct (T1-T2)')


In [ ]:
# SHAP plot on saved model
cb_ul_model = CatBoostRegressor()
cb_ul_model.load_model(os.path.join(MODEL_DIR, 'cb_ul_model.cbm'))
cb_ul_X = joblib.load(os.path.join(MODEL_DIR, 'cb_ul_X.pkl'))

print('\nSHAP — CatBoost (pain_under_load_reduction, T1-T2)')
cb_ul_shap = model.plot_shap_regressor(
    cb_ul_model, cb_ul_X, 'CatBoost — pain_under_load_reduction (T1-T2)')


## Model 2 - RENT, Nested CV + Optuna Hyperparameter Tuning

## Model 3 - RENT, Nested CV + Optuna Hyperparameter Tuning